# Iterator Utilities API Reference

Developer-facing statements defined in `libs/core/langchain_core/utils/iter.py`.

# `NoLock`

Synchronous context manager with the lock interface but no synchronization.

## Methods

### `__enter__`

Performs no action and returns `None`.

```python
__enter__(
    self,
) -> None
```

### `__exit__`

Returns `False`, so exceptions are not suppressed.

```python
__exit__(
    self,
    exc_type: type[BaseException] | None, # Exception class, when present
    exc_val: BaseException | None, # Exception instance, when present
    exc_tb: TracebackType | None, # Exception traceback, when present
) -> Literal[False] # Always False
```

---

# `Tee: Generic[T]`

Splits one synchronous iterator into multiple child iterators that yield the same items in the same order.

Each child can advance independently. Items retrieved by a more advanced child are buffered until the remaining active children consume them.

## Constructor

```python
Tee(
    iterable: Iterator[T], # Iterator to split
    n: int = 2, # Number of child iterators to create
    *,
    lock: AbstractContextManager[Any] | None = None, # Context-manager lock used to synchronize source access
) -> None
```

When `lock` is omitted, `NoLock` is used. A real synchronous lock can be supplied when access to the source iterator must be serialized.

## Methods

### `__len__`

Returns the number of child iterators.

```python
__len__(
    self,
) -> int # Number of children
```

### `__getitem__`

Returns one child iterator by index or multiple child iterators by slice.

```python
@overload
__getitem__(
    self,
    item: int, # Child index
) -> Iterator[T]
```

```python
@overload
__getitem__(
    self,
    item: slice, # Child slice
) -> tuple[Iterator[T], ...]
```

### `__iter__`

Iterates over the child iterators, allowing unpacking and normal iteration over the `Tee` container.

```python
__iter__(
    self,
) -> Iterator[Iterator[T]] # Child iterators
```

### `__enter__`

Returns the current `Tee` instance.

```python
__enter__(
    self,
) -> "Tee[T]" # Current instance
```

### `__exit__`

Closes all child iterators and returns `False`, so exceptions are not suppressed.

```python
__exit__(
    self,
    exc_type: type[BaseException] | None, # Exception class, when present
    exc_val: BaseException | None, # Exception instance, when present
    exc_tb: TracebackType | None, # Exception traceback, when present
) -> Literal[False] # Always False
```

### `close`

Closes every child iterator.

```python
close(
    self,
) -> None
```

When the final active child closes or finishes, the shared source iterator is also closed when it provides `close()`.

---

# `safetee`

Alias for `Tee`.

```python
safetee = Tee
```

---

# `batch_iterate`

Groups values from a synchronous iterable into lists.

```python
batch_iterate(
    size: int | None, # Maximum number of items in each batch, or None for one batch
    iterable: Iterable[T], # Iterable to batch
) -> Iterator[list[T]] # Full batches followed by a final partial batch when needed
```

When `size` is `None`, all remaining items are returned in one batch. An empty iterable produces no batches.

In [ ]:
# 1. Divide an iterable into batches
from langchain_core.utils.iter import batch_iterate # Import the synchronous batching utility


numbers = range(1, 11) # Create numbers from 1 through 10

batches = batch_iterate( # Create batches from the numbers
    size=3, # Place at most three items in each batch
    iterable=numbers, # Provide the iterable to divide
) # Finish creating the batch iterator

for batch in batches: # Iterate through each generated batch
    print(batch) # Display the current batch

In [ ]:
# 2. Put all items into one batch
from langchain_core.utils.iter import batch_iterate # Import the batching utility


values = ["Python", "SQL", "LangChain", "Pandas"] # Create sample values

single_batch = batch_iterate( # Create one batch containing all remaining items
    size=None, # Disable the batch-size limit
    iterable=values, # Provide the values to batch
) # Finish creating the iterator

for batch in single_batch: # Iterate through the generated batch
    print(batch) # Display all values in one list

In [ ]:
# 3. Split one iterator using Tee
from langchain_core.utils.iter import Tee # Import the iterator-splitting class


source = iter([10, 20, 30, 40]) # Create the original iterator

tee = Tee( # Split the original iterator
    source, # Provide the iterator to split
    n=2, # Create two independent child iterators
) # Finish creating the Tee object

first_iterator, second_iterator = tee # Unpack the two child iterators

print(next(first_iterator)) # Read 10 from the first child
print(next(first_iterator)) # Read 20 from the first child
print(next(second_iterator)) # Read the buffered 10 from the second child
print(next(second_iterator)) # Read the buffered 20 from the second child
print(next(second_iterator)) # Read 30 from the shared source
print(next(first_iterator)) # Read the buffered 30 from the first child

tee.close() # Close all child iterators
# Each child receives the same items in the same order, even when one child moves ahead.

In [ ]:
# 4. Access child iterators by index and slice
from langchain_core.utils.iter import Tee # Import the Tee class


tee = Tee(iter(["A", "B", "C"]), n=3) # Create three child iterators

first_child = tee[0] # Retrieve the first child by index
remaining_children = tee[1:] # Retrieve the remaining children by slice

print(next(first_child)) # Read A from the first child
print(next(remaining_children[0])) # Read the buffered A from the second child
print(next(remaining_children[1])) # Read the buffered A from the third child

tee.close() # Close every child iterator

In [ ]:
# 5. Use the safetee alias with a lock
from threading import Lock # Import a real synchronization lock

from langchain_core.utils.iter import safetee # Import the alias for Tee


source = iter(["document-1", "document-2", "document-3"]) # Create a shared document iterator
lock = Lock() # Create a lock for synchronized source access

with safetee(source, n=2, lock=lock) as split_iterators: # Split and automatically close the iterator
    indexing_iterator, logging_iterator = split_iterators # Unpack the child iterators

    print("Indexing:", next(indexing_iterator)) # Read the first document for indexing
    print("Indexing:", next(indexing_iterator)) # Read the second document for indexing

    print("Logging:", next(logging_iterator)) # Read the buffered first document for logging
    print("Logging:", next(logging_iterator)) # Read the buffered second document for logging

In [ ]:
# 6. Demonstrate NoLock
from langchain_core.utils.iter import NoLock # Import the no-operation context manager


no_lock = NoLock() # Create a lock-compatible context manager

with no_lock: # Enter the context without performing synchronization
    print("This code runs without acquiring a real lock.") # Execute code normally